# 4.0 - Inferencia

Genera las predicciones de `SalePrice` para `test.csv` partiendo de los **datos crudos**:
1. Carga el modelo ganador (`models/modelo_ganador.joblib`), que ya contiene el pipeline de ingeniería de características y el modelo.
2. Construye un **pipeline de inferencia** con todo el flujo: selección de features → ingeniería de características → modelo.
3. Predice `test.csv` y guarda las predicciones en formato Kaggle (`Id`, `SalePrice`).
4. Guarda el pipeline de inferencia en `models/pipeline_inferencia.joblib`.

## 1. Configuración

Importa las librerías y las rutas del proyecto.

In [1]:
%load_ext autoreload
%autoreload 2

import joblib
import numpy as np
import pandas as pd
import yaml

from feature_engine.preprocessing import MatchVariables
from sklearn.pipeline import Pipeline

from proyecto_pd.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

pd.set_option("display.max_columns", 100)

2026-10-29 20:04:20.584 | INFO     | proyecto_pd.config:<module>:11 - PROJ_ROOT path is: C:\Users\doty_\Documents\PD\Proyecto\proyecto_pd


## 2. Carga de datos crudos de test

Lee `test.csv` completo, con sus 80 columnas originales y sin ninguna transformación. `Id` queda como índice para identificar cada predicción.

In [2]:
test = pd.read_csv(RAW_DATA_DIR / "test.csv", index_col="Id")

print(f"test: {test.shape}")
test.head()

test: (1459, 79)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0,TA,TA,PConc,TA,TA,No,GLQ,602.0,Unf,0.0,324.0,926.0,GasA,Ex,Y,SBrkr,926,678,0,1604,0.0,0.0,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,1998.0,Fin,2.0,470.0,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,PConc,Gd,TA,No,ALQ,263.0,Unf,0.0,1017.0,1280.0,GasA,Ex,Y,SBrkr,1280,0,0,1280,0.0,0.0,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,1992.0,RFn,2.0,506.0,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


## 3. Carga del modelo ganador

Carga el pipeline guardado en el notebook 3.0. Ya está ajustado con train + validación y contiene:
- `features`: el pipeline de ingeniería de características (imputación, codificación, outliers, transformaciones y escalado).
- `modelo`: el modelo ganador del torneo.

In [3]:
modelo_ganador = joblib.load(MODELS_DIR / "modelo_ganador.joblib")

for nombre, paso in modelo_ganador.steps:
    print(f"{nombre}: {type(paso).__name__}")
modelo_ganador

features: Pipeline
modelo: GradientBoostingRegressor


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('descartar_faltantes', ...), ('imputar_mediana', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`

## 4. Pipeline de inferencia

Agrega al inicio un paso de **selección de features** con `MatchVariables` de feature-engine. Este paso se ajusta con las columnas de `MODEL_FEATURES` de los datos de entrenamiento, y en `transform`:
- Elimina las columnas crudas que el modelo no usa.
- Ordena las 30 features igual que en el entrenamiento.
- Si falta alguna columna en los datos nuevos, la agrega como faltante para que la imputación la trate.

El resultado es un único `Pipeline`: **datos crudos → selección → ingeniería de características → modelo → predicción**.

In [4]:
with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    MODEL_FEATURES = yaml.safe_load(f)["MODEL_FEATURES"]

# Solo se aprenden los nombres de las columnas, no estadísticas
columnas_entrenamiento = pd.read_csv(RAW_DATA_DIR / "train.csv", index_col="Id")[MODEL_FEATURES]
seleccion_features = MatchVariables(missing_values="ignore", verbose=False).fit(columnas_entrenamiento)

pipeline_inferencia = Pipeline([("seleccionar_features", seleccion_features)] + modelo_ganador.steps)

for nombre, paso in pipeline_inferencia.steps:
    print(f"{nombre}: {type(paso).__name__}")
pipeline_inferencia

seleccionar_features: MatchVariables
features: Pipeline
modelo: GradientBoostingRegressor


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('seleccionar_features', ...), ('features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,missing_values,'ignore'
,verbose,False
,fill_value,nan
,match_dtypes,False


## 5. Predicciones sobre test

Pasa los datos crudos de test por el pipeline de inferencia. Las predicciones salen en dólares, porque el modelo se entrenó con `SalePrice` sin transformar.

Se verifica que:
- Hay una predicción por casa.
- No hay valores faltantes ni negativos.
- El resultado coincide con aplicar el modelo ganador directamente sobre las 30 features.

In [5]:
predicciones = pd.DataFrame(
    {"SalePrice": pipeline_inferencia.predict(test)},
    index=test.index,
).reset_index()

print(f"Predicciones: {len(predicciones)} | casas en test: {len(test)}")
print(f"Faltantes: {predicciones['SalePrice'].isna().sum()} | negativas: {(predicciones['SalePrice'] < 0).sum()}")
print(f"Coincide con el modelo directo: {np.allclose(predicciones['SalePrice'], modelo_ganador.predict(test[MODEL_FEATURES]))}")

predicciones["SalePrice"].describe().round(0)

Predicciones: 1459 | casas en test: 1459
Faltantes: 0 | negativas: 0
Coincide con el modelo directo: True


count      1459.0
mean     179119.0
std       77942.0
min       38408.0
25%      129001.0
50%      157929.0
75%      208930.0
max      559003.0
Name: SalePrice, dtype: float64

## 6. Guardar predicciones y pipeline de inferencia

- `data/processed/predicciones_test.csv`: predicciones en formato Kaggle (`Id`, `SalePrice`), con las mismas columnas que `sample_submission.csv`.
- `models/pipeline_inferencia.joblib`: pipeline completo, para predecir datos crudos nuevos con `joblib.load(...).predict(datos)`.

Al final se recarga el pipeline guardado y se comprueba que da las mismas predicciones.

In [6]:
predicciones.to_csv(PROCESSED_DATA_DIR / "predicciones_test.csv", index=False)
joblib.dump(pipeline_inferencia, MODELS_DIR / "pipeline_inferencia.joblib")

muestra = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")
recargado = joblib.load(MODELS_DIR / "pipeline_inferencia.joblib")

print(f"Mismas columnas que sample_submission: {list(predicciones.columns) == list(muestra.columns)}")
print(f"Mismos Id que sample_submission: {predicciones['Id'].equals(muestra['Id'])}")
print(f"Pipeline recargado da las mismas predicciones: {np.allclose(recargado.predict(test), predicciones['SalePrice'])}")
predicciones.head()

Mismas columnas que sample_submission: True
Mismos Id que sample_submission: True
Pipeline recargado da las mismas predicciones: True


,Id,SalePrice
0,1461,121190.379508
1,1462,154064.915010
2,1463,179182.759398
3,1464,187137.773137
4,1465,193887.277683
